# 📱 대학생 스마트폰 사용 패턴 × 우울증(PHQ-9) 분석
**데이터**: StudentLife, Dartmouth University 2013  
**방식**: 전체 압축 해제 없이 필요한 파일만 직접 추출  

> ▶ **셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)**

## STEP 1. 라이브러리 설치 및 한글 폰트

In [ ]:
import subprocess, sys

# 한글 폰트
subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats as sp_stats
import warnings, io, tarfile, urllib.request, os
warnings.filterwarnings('ignore')

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('✅ 환경 준비 완료')

## STEP 2. 필요한 파일만 스트리밍 추출
> 전체 압축 해제 없이 `PHQ-9.csv`와 `phonelock` 파일만 메모리로 읽어옵니다.  
> ⏱ 약 3~7분 소요 (5GB 파일을 스트리밍으로 읽는 방식)

In [ ]:
URL = 'https://studentlife.cs.dartmouth.edu/dataset/dataset.tar.bz2'

# 추출할 파일 패턴
TARGET_PATTERNS = [
    'survey/PHQ-9.csv',
    'sensing/phonelock/',
]

extracted = {}   # { 파일명: DataFrame }

def is_target(name):
    for pat in TARGET_PATTERNS:
        if pat in name and name.endswith('.csv'):
            return True
    return False

print('🔄 스트리밍 다운로드 시작... (잠시 기다려 주세요)')
print('   필요한 파일만 추출하므로 전체 다운로드보다 빠릅니다.')

with urllib.request.urlopen(URL) as response:
    with tarfile.open(fileobj=response, mode='r|bz2') as tar:
        for member in tar:
            if member.isfile() and is_target(member.name):
                f = tar.extractfile(member)
                if f:
                    short = member.name.split('/')[-1]
                    try:
                        extracted[short] = pd.read_csv(io.BytesIO(f.read()))
                        print(f'  ✔ {short} ({len(extracted[short])}행)')
                    except Exception as e:
                        print(f'  ⚠ {short} 읽기 실패: {e}')

print(f'\n✅ 추출 완료 — 총 {len(extracted)}개 파일')

## STEP 3. PHQ-9 전처리

In [ ]:
phq_raw = extracted.get('PHQ-9.csv')
if phq_raw is None:
    raise ValueError('PHQ-9.csv를 찾지 못했습니다. STEP 2를 다시 실행하세요.')

print('원본 컬럼:', phq_raw.columns.tolist())
print('크기:', phq_raw.shape)
phq_raw.head()

In [ ]:
phq = phq_raw.copy()
phq.columns = [c.lower().strip() for c in phq.columns]

# uid 컬럼 확인
uid_col = next((c for c in phq.columns if 'uid' in c), None)
if uid_col and uid_col != 'uid':
    phq = phq.rename(columns={uid_col: 'uid'})

# PHQ-9 문항 컬럼 (uid 제외 숫자형)
num_cols = phq.select_dtypes(include='number').columns.tolist()
item_cols = [c for c in num_cols if c != 'uid']
print('PHQ-9 문항 컬럼:', item_cols)

# 총점 계산
phq['phq9_total'] = phq[item_cols].sum(axis=1)

# 결측치 제거
phq = phq.dropna(subset=['phq9_total'])

# 이상치 제거 (PHQ-9 범위: 0~27)
phq = phq[(phq['phq9_total'] >= 0) & (phq['phq9_total'] <= 27)]

# 심각도 분류
def severity(s):
    if s <= 4:    return '없음(0-4)'
    elif s <= 9:  return '경미(5-9)'
    elif s <= 14: return '중등도(10-14)'
    elif s <= 19: return '중증(15-19)'
    else:         return '최중증(20-27)'

phq['severity'] = phq['phq9_total'].apply(severity)
phq_final = phq[['uid', 'phq9_total', 'severity']].reset_index(drop=True)

print(f'\n✅ PHQ-9 전처리 완료: {len(phq_final)}명')
phq_final.head()

## STEP 4. 스마트폰 사용 패턴 전처리

In [ ]:
# phonelock 파일들 합치기
lock_files = {k: v for k, v in extracted.items() if k.startswith('phonelock_u')}
print(f'phonelock 파일 수: {len(lock_files)}개')

lock_list = []
for fname, df in lock_files.items():
    try:
        uid = int(fname.replace('phonelock_u', '').replace('.csv', ''))
        df = df.copy()
        df.columns = ['time', 'lock_status'] if len(df.columns) == 2 else df.columns
        df['uid'] = uid
        lock_list.append(df)
    except:
        pass

if not lock_list:
    print('⚠ phonelock 파일 없음 — 컬럼명 자동 탐색 시도')
    # 컬럼이 다를 수 있으므로 raw 출력
    for k, v in list(extracted.items())[:3]:
        print(k, v.columns.tolist())
else:
    lock_raw = pd.concat(lock_list, ignore_index=True)
    print(f'✅ 합산: {len(lock_raw):,}행 / {lock_raw.uid.nunique()}명')
    print('lock_status 고유값:', lock_raw['lock_status'].unique())
    lock_raw.head()

In [ ]:
# timestamp → 날짜 변환
lock_raw['datetime'] = pd.to_datetime(lock_raw['time'], unit='s', errors='coerce')
lock_raw = lock_raw.dropna(subset=['datetime'])
lock_raw['date'] = lock_raw['datetime'].dt.date

# 잠금 해제(unlock) 이벤트만 선택
unlock = lock_raw[lock_raw['lock_status'].astype(str).isin(['1', 'unlock', 'UNLOCK'])]
print(f'unlock 이벤트: {len(unlock):,}건')

# 참가자별 일평균 사용 횟수
daily = unlock.groupby(['uid', 'date']).size().reset_index(name='unlock_count')
usage = daily.groupby('uid')['unlock_count'].agg(
    mean_unlock='mean',
    std_unlock='std',
    obs_days='count'
).reset_index().round(2)

# IQR 이상치 표시
Q1, Q3 = usage['mean_unlock'].quantile([0.25, 0.75])
IQR = Q3 - Q1
usage['is_outlier'] = (usage['mean_unlock'] < Q1 - 1.5*IQR) | (usage['mean_unlock'] > Q3 + 1.5*IQR)

print(f'✅ 사용 패턴 집계: {len(usage)}명  |  이상치: {usage.is_outlier.sum()}명')
usage.head()

## STEP 5. 병합 및 기술통계

In [ ]:
df = pd.merge(phq_final, usage, on='uid', how='inner')
print(f'최종 분석 대상: {len(df)}명\n')

# 기술통계
desc = df[['phq9_total', 'mean_unlock']].describe().round(2)
desc.index = ['N', '평균', '표준편차', '최솟값', 'Q1', '중앙값', 'Q3', '최댓값']
desc.columns = ['PHQ-9 총점', '일평균 잠금해제 횟수']
print(desc.to_string())

print('\n📋 PHQ-9 심각도 분포')
for s, n in df['severity'].value_counts().items():
    pct = n / len(df) * 100
    print(f'  {s}: {n}명 ({pct:.1f}%)')

## STEP 6. 시각화 (4종)

In [ ]:
SEV_ORDER  = ['없음(0-4)', '경미(5-9)', '중등도(10-14)', '중증(15-19)', '최중증(20-27)']
SEV_COLORS = {'없음(0-4)':'#2ECC71','경미(5-9)':'#F1C40F',
              '중등도(10-14)':'#E67E22','중증(15-19)':'#E74C3C','최중증(20-27)':'#8E44AD'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('StudentLife: 스마트폰 사용 패턴 × PHQ-9 분석', fontsize=16, fontweight='bold', y=0.98)

# ── 그래프 1: PHQ-9 분포 ──────────────────────────────────
ax = axes[0, 0]
ax.hist(df['phq9_total'], bins=14, color='#4A90D9', edgecolor='white', linewidth=0.8)
ax.axvline(df['phq9_total'].mean(),   color='red',    ls='--', lw=1.8, label=f'평균 {df["phq9_total"].mean():.1f}')
ax.axvline(df['phq9_total'].median(), color='orange', ls='--', lw=1.8, label=f'중앙값 {df["phq9_total"].median():.1f}')
ax.set_xlabel('PHQ-9 총점'); ax.set_ylabel('참가자 수')
ax.set_title('① PHQ-9 점수 분포'); ax.legend()

# ── 그래프 2: 심각도별 막대 ───────────────────────────────
ax = axes[0, 1]
present = [s for s in SEV_ORDER if s in df['severity'].values]
counts  = [df['severity'].value_counts().get(s, 0) for s in present]
colors  = [SEV_COLORS[s] for s in present]
bars = ax.bar(range(len(present)), counts, color=colors, edgecolor='white')
ax.set_xticks(range(len(present)))
ax.set_xticklabels(present, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('참가자 수'); ax.set_title('② 우울증 심각도 분포')
for b, c in zip(bars, counts):
    if c: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, str(c), ha='center', va='bottom')

# ── 그래프 3: 산점도 + 회귀선 ────────────────────────────
ax = axes[1, 0]
x, y = df['mean_unlock'], df['phq9_total']
slope, intercept, r, p, _ = sp_stats.linregress(x, y)
c_list = df['severity'].map(SEV_COLORS)
ax.scatter(x, y, c=c_list, s=55, alpha=0.75, edgecolors='white', lw=0.5)
xr = np.linspace(x.min(), x.max(), 100)
ax.plot(xr, slope*xr+intercept, color='navy', lw=2, label=f'r={r:.3f}  p={p:.3f}')
ax.set_xlabel('일평균 잠금해제 횟수'); ax.set_ylabel('PHQ-9 총점')
ax.set_title('③ 산점도 + 선형 회귀선')
ax.legend()
sig = '★ 유의미 (p<0.05)' if p < 0.05 else '비유의 (p≥0.05)'
ax.text(0.97, 0.05, sig, transform=ax.transAxes, ha='right', fontsize=9,
        color='navy' if p < 0.05 else 'gray')

# ── 그래프 4: 심각도별 박스플롯 ──────────────────────────
ax = axes[1, 1]
data_bp = [df[df['severity']==s]['mean_unlock'].values for s in present]
bp = ax.boxplot(data_bp, patch_artist=True, medianprops=dict(color='black', lw=2),
                flierprops=dict(marker='o', markersize=4, color='gray'))
for patch, s in zip(bp['boxes'], present):
    patch.set_facecolor(SEV_COLORS[s]); patch.set_alpha(0.75)
ax.set_xticks(range(1, len(present)+1))
ax.set_xticklabels(present, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('일평균 잠금해제 횟수'); ax.set_title('④ 심각도별 스마트폰 사용 빈도')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('분석결과_시각화.png', bbox_inches='tight')
plt.show()
print(f'\n📌 상관계수 r={r:.4f}  |  p={p:.4f}  |  {sig}')

In [ ]:
# ── 상관관계 히트맵 ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[['phq9_total','mean_unlock','std_unlock','obs_days']].rename(columns={
    'phq9_total':'PHQ-9 총점','mean_unlock':'평균 사용 횟수',
    'std_unlock':'사용 변동성','obs_days':'관측 일수'
}).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, mask=mask, linewidths=0.5, square=True,
            annot_kws={'size':11}, ax=ax)
ax.set_title('상관관계 히트맵', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('상관관계_히트맵.png', bbox_inches='tight')
plt.show()

## STEP 7. 결과 저장

In [ ]:
df.to_csv('분석데이터_최종.csv', index=False, encoding='utf-8-sig')
desc.to_csv('기술통계.csv', encoding='utf-8-sig')

print('저장 완료:')
for f in ['분석데이터_최종.csv','기술통계.csv','분석결과_시각화.png','상관관계_히트맵.png']:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  ✔ {f}  ({size:,} bytes)')

print('\n✅ 모든 분석 완료!')

---
### 결과 요약
| 그래프 | 내용 |
|--------|------|
| ① PHQ-9 점수 분포 | 평균·중앙값 표시된 히스토그램 |
| ② 심각도 분포 | 없음~최중증 5단계 막대그래프 |
| ③ 산점도+회귀선 | r값·p값·유의성 자동 표시 |
| ④ 심각도별 박스플롯 | 심각도별 스마트폰 사용 비교 |
| 히트맵 | 변수 간 상관관계 한눈에 |

**다음 단계로 확장하려면:**  
- 회귀분석 심화 → `statsmodels` OLS  
- 충전 데이터 추가 → `sensing/phonecharge/`  
- 시계열(10주) 추이 분석